In [15]:
import pandas as pd
import os

def get_ticker_from_filename(filename):
    """Extract ticker from filename (e.g., 'AMZN_updated.csv' -> 'AMZN')."""
    # Extract basename (e.g., 'AMZN_updated.csv' from 'S&P Updated/AMZN_updated.csv')
    basename = os.path.basename(filename)
    # Split on '_' and take the first part (e.g., 'AMZN' from 'AMZN_updated.csv')
    ticker = basename.split('_')[0]
    return ticker


In [17]:
import glob

try:
    # Get all CSVs in S&P Updated/
    csv_files = glob.glob('S&P Updated/*.csv')
    dfs = []
    
    # Load each CSV and add ticker column
    for csv_file in csv_files:
        df = pd.read_csv(csv_file)
        df['ticker'] = get_ticker_from_filename(csv_file)
        dfs.append(df)
        print(f"Loaded {csv_file}: {len(df)} rows, Ticker: {df['ticker'].iloc[0]}")
    
    # Combine all CSVs
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Check for duplicates based on 'id'
    duplicate_count = combined_df.duplicated(subset='id').sum()
    if duplicate_count > 0:
        print(f"Warning: Found {duplicate_count} duplicate IDs. Removing duplicates...")
        combined_df = combined_df.drop_duplicates(subset='id', keep='first')
    
    # Ensure all expected columns
    expected_columns = [
        'ticker', 'type', 'id', 'parent_post_id', 'subreddit', 'title', 'text',
        'url', 'upvotes', 'created_utc', 'author', 'word_count', 'author_created_utc'
    ]
    for col in expected_columns:
        if col not in combined_df.columns:
            combined_df[col] = ''
    combined_df = combined_df[expected_columns]
    
    # Save combined CSV
    output_file = 'S&P Updated/sp100.csv'
    combined_df.to_csv(output_file, index=False, encoding='utf-8')
    
    # Print summary
    print(f"\nCombined {len(csv_files)} CSVs into {output_file}")
    print(f"Total rows: {len(combined_df)}")
    print(f"Unique tickers: {combined_df['ticker'].nunique()}")
    print(f"Columns: {list(combined_df.columns)}")
    
except Exception as e:
    print(f"Error combining files: {e}")

Loaded S&P Updated\AAPL_updated.csv: 8350 rows, Ticker: AAPL
Loaded S&P Updated\ABBV_updated.csv: 122 rows, Ticker: ABBV
Loaded S&P Updated\ABT_updated.csv: 333 rows, Ticker: ABT
Loaded S&P Updated\ACN_updated.csv: 76 rows, Ticker: ACN
Loaded S&P Updated\ADBE_updated.csv: 689 rows, Ticker: ADBE
Loaded S&P Updated\AIG_updated.csv: 203 rows, Ticker: AIG
Loaded S&P Updated\AMD_updated.csv: 5184 rows, Ticker: AMD
Loaded S&P Updated\AMGN_updated.csv: 43 rows, Ticker: AMGN
Loaded S&P Updated\AMT_updated.csv: 330 rows, Ticker: AMT
Loaded S&P Updated\AMZN_updated.csv: 4853 rows, Ticker: AMZN
Loaded S&P Updated\AVGO_updated.csv: 769 rows, Ticker: AVGO
Loaded S&P Updated\AXP_updated.csv: 489 rows, Ticker: AXP
Loaded S&P Updated\BAC_updated.csv: 1053 rows, Ticker: BAC
Loaded S&P Updated\BA_updated.csv: 985 rows, Ticker: BA
Loaded S&P Updated\BKNG_updated.csv: 584 rows, Ticker: BKNG
Loaded S&P Updated\BK_updated.csv: 49 rows, Ticker: BK
Loaded S&P Updated\BLK_updated.csv: 650 rows, Ticker: BLK
Loa

In [5]:
import pandas as pd

# Specify file paths
sentiment_file = 'sp100_sentiment.csv'
weights_file = 'Weights-05_25_2025.csv'
output_file = 'sp100_sentiment_with_sector.csv'

try:
    # Load sentiment and weights CSVs
    df_sentiment = pd.read_csv(sentiment_file)
    df_weights = pd.read_csv(weights_file)
    
    # Standardize ticker columns (e.g., convert to uppercase)
    df_sentiment['ticker'] = df_sentiment['ticker'].str.upper()
    df_weights['Ticker'] = df_weights['Ticker'].str.upper()
    
    # Merge to add sector column
    df_merged = df_sentiment.merge(
        df_weights[['Ticker', 'Sector']],
        left_on='ticker',
        right_on='Ticker',
        how='left'
    )
    
    # Drop the redundant Ticker column from weights
    df_merged = df_merged.drop(columns=['Ticker'])
    
    # Rename Sector to sector
    df_merged = df_merged.rename(columns={'Sector': 'sector'})
    
    # Reorder columns to place sector after ticker
    columns = ['ticker', 'sector'] + [col for col in df_sentiment.columns if col != 'ticker']
    df_merged = df_merged[columns]
    
    # Identify unmatched tickers
    unmatched_tickers = df_merged[df_merged['sector'].isna()]['tficker'].unique()
    
    # Save the new dataset
    df_merged.to_csv(output_file, index=False)
    print(f"Saved new dataset with sector column to {output_file}")
    
    # Print summary
    print("\nSample of the new dataset (first 5 rows):")
    print(df_merged[['ticker', 'sector'] + df_sentiment.columns[1:].tolist()].head())
    print(f"\nTotal rows in new dataset: {len(df_merged)}")
    print(f"Unmatched tickers (no sector assigned): {len(unmatched_tickers)}")
    if len(unmatched_tickers) > 0:
        print(f"List of unmatched tickers: {unmatched_tickers}")
    
except Exception as e:
    print(f"Error processing files: {e}")

Saved new dataset with sector column to sp100_sentiment_with_sector.csv

Sample of the new dataset (first 5 rows):
  ticker                  sector     type       id parent_post_id  \
0   AAPL  Information Technology     post  18ecxrr            NaN   
1   AAPL  Information Technology  comment  kcmvg9s        18ecxrr   
2   AAPL  Information Technology  comment  kcmxlan        18ecxrr   
3   AAPL  Information Technology  comment  kcn7z6v        18ecxrr   
4   AAPL  Information Technology  comment  kcnfgkx        18ecxrr   

        subreddit                                              title  \
0  wallstreetbets  Apple's head of iphone and Apple watch design ...   
1  wallstreetbets                                                NaN   
2  wallstreetbets                                                NaN   
3  wallstreetbets                                                NaN   
4  wallstreetbets                                                NaN   

                                     